# Cyberbullying model using LSTM

In [1]:
import pandas as pd
import numpy as np
import string
import nltk
from nltk.stem import WordNetLemmatizer
from nltk.corpus import stopwords
from nltk.corpus import wordnet
from nltk import pos_tag
from nltk.tokenize import word_tokenize
from sklearn.feature_extraction.text import CountVectorizer

In [2]:
from tensorflow.keras.preprocessing.text import Tokenizer
from keras.models import Model
from keras.layers import Dense, Input, Dropout, LSTM, Activation
from keras.layers.embeddings import Embedding
from keras.preprocessing import sequence
from keras.initializers import glorot_uniform

ImportError: cannot import name 'runtime_version' from 'google.protobuf' (c:\users\nlhri\appdata\local\programs\python\python39\lib\site-packages\google\protobuf\__init__.py)

## Preprocessing the dataset

In [ ]:
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))
stop_words.update(list(string.punctuation))

In [ ]:
df = pd.read_csv("cyberbullying_tweets.csv")
df.head()

In [ ]:
messages = df['tweet_text']
y = df['cyberbullying_type']

In [ ]:
df['tweet_text']

In [ ]:
def get_simple_pos(tag) :
    if tag.startswith('J') :
        return wordnet.ADJ
    elif tag.startswith('V') :
        return wordnet.VERB
    elif tag.startswith('N') :
        return wordnet.NOUN
    elif tag.startswith('R') :
        return wordnet.ADV
    else:
        return wordnet.NOUN

def clean_text(review) :
    global max_len
    words = word_tokenize(review)
    output_words = []
    for word in words :
        if word.lower() not in stop_words :
            pos = pos_tag([word])
            clean_word = lemmatizer.lemmatize(word,pos = get_simple_pos(pos[0][1]))
            output_words.append(clean_word.lower())
    max_len = max(max_len, len(output_words))
    return " ".join(output_words)

In [ ]:
max_len = 0

In [ ]:
print(messages[0])
messages = [clean_text(message) for message in messages]
print(messages[0])

In [ ]:
def read_glove_vecs(glove_file):
    with open(glove_file, 'r', encoding="utf8") as file:
        word_to_vec_map = {}
        word_to_index = {}
        index_to_word = {}
        index = 0
        for line in file:
            line = line.strip().split()
            curr_word = line[0]
            word_to_index[curr_word] = index
            index_to_word[index] = curr_word
            word_to_vec_map[curr_word] = np.array(line[1:], dtype=np.float64)
            index += 1
    return word_to_index, index_to_word, word_to_vec_map

In [ ]:
word_to_index, index_to_word, word_to_vec_map = read_glove_vecs('glove.6B.50d.txt')

In [ ]:
def sentences_to_indices(X, word_to_index, max_len):
    m = len(X)
    X_indices = np.zeros((m, max_len))
    for i in range(m):
        sentence_words = [w.lower() for w in X[i].split()]
        j = 0
        for word in sentence_words:
            if word in word_to_index:
                X_indices[i, j] = word_to_index[word]
            j += 1
    return X_indices

## The LSTM and CNN model

In [ ]:
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Embedding, LSTM, Conv1D, MaxPooling1D, GlobalMaxPooling1D, concatenate, Dense, Dropout
from tensorflow.keras.utils import to_categorical

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

# Encode target labels
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)
y_cat = to_categorical(y_encoded)

# Split data
X = sentences_to_indices(messages, word_to_index, max_len)
X_train, X_test, y_train, y_test = train_test_split(X, y_cat, test_size=0.2, random_state=42)

# Embedding layer preparation
vocab_len = len(word_to_index)
emb_dim = 50

embedding_matrix = np.zeros((vocab_len, emb_dim))
for word, index in word_to_index.items():
    embedding_vector = word_to_vec_map.get(word)
    if embedding_vector is not None:
        embedding_matrix[index] = embedding_vector

embedding_layer = Embedding(input_dim=vocab_len,
                            output_dim=emb_dim,
                            weights=[embedding_matrix],
                            input_length=max_len,
                            trainable=False)

# Define input
input_layer = Input(shape=(max_len,))

# Embedding
embedded_sequences = embedding_layer(input_layer)

# LSTM Branch
lstm_branch = LSTM(64, return_sequences=True)(embedded_sequences)
lstm_branch = GlobalMaxPooling1D()(lstm_branch)

# CNN Branch
cnn_branch = Conv1D(filters=128, kernel_size=5, activation='relu')(embedded_sequences)
cnn_branch = MaxPooling1D(pool_size=2)(cnn_branch)
cnn_branch = GlobalMaxPooling1D()(cnn_branch)

# Concatenate both branches
merged = concatenate([lstm_branch, cnn_branch])
merged = Dense(128, activation='relu')(merged)
merged = Dropout(0.5)(merged)
output_layer = Dense(y_cat.shape[1], activation='softmax')(merged)

# Define and compile model
model = Model(inputs=input_layer, outputs=output_layer)
model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

# Summary
model.summary()

# Train
model.fit(X_train, y_train, epochs=10, batch_size=64, validation_data=(X_test, y_test))


In [ ]:
loss, accuracy = model.evaluate(X_test, y_test)
print(f"Test Accuracy: {accuracy * 100:.2f}%")


## Accuracy of LSTM: 88.25%

In [ ]:
model.save("model.h5")

## Predictions

In [ ]:
text = "suck it"
text = [clean_text(text)]
text

In [ ]:
text = sentences_to_indices(text, word_to_index, max_len)

In [ ]:
text

In [ ]:
model.predict(text)[0][0]

## Extras

In [ ]:
import pickle

In [ ]:
pickle.dump(word_to_index, open('word_to_index.pkl', 'wb'))